In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'tensorflow', 'keras', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'keras': 'keras==3.14.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "5d733eab198ad58f23ceee3f1550014385366ece"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/5d733eab198ad58f23ceee3f1550014385366ece/d2l"
for name in ('__init__.py', 'tensorflow.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Numerics: Dtypes and Mixed Precision

Every tensor carries three attributes: a shape, a device, and a *dtype*, the
numeric format of its entries. So far we have left the dtype at its default,
32-bit floating point, and nothing forced us to look at it. That grace period
is over. Modern accelerators multiply 16-bit matrices several times faster
than 32-bit ones, in half the memory, and essentially all serious training now
chooses numeric formats deliberately. This section covers what a builder needs:
which formats exist, what each one can and cannot represent, how dtypes combine,
and the standard recipe (*mixed precision*) for training in 16 bits without
giving up 32-bit accuracy. For the anatomy of a floating-point number (sign,
exponent, mantissa) see that section;
here we ask the practical questions: when does it break, and which switch do
I flip.

In [ ]:
import ml_dtypes
import time
import tensorflow as tf
from d2l import tensorflow as d2l

## The Dtype Zoo

Half-precision (`float16`, "fp16") sounds like a free lunch: half the
bytes of fp32, and the format that accelerator hardware sped up first. Here
is the catch. The largest number fp16 can represent is 65504. Square a value
of 300, which is nothing exotic (an unnormalized logit, an intermediate in a
variance computation), and you have already left the representable range:

In [ ]:
x = tf.constant(300.0)
tf.cast(x, tf.float16)**2, tf.cast(x, tf.bfloat16)**2

fp16 overflows to `inf`. The second format, `bfloat16` ("bf16", brain
floating point), returns 90112: wrong in the fourth digit, since the exact
square is 90000, but finite. The two formats spend the same 16 bits
differently. fp16 uses 5 exponent bits and 10 mantissa bits: fine-grained
steps, tiny range. bf16 keeps fp32's full 8 exponent bits and pays with a
7-bit mantissa: fp32's range, coarse steps. The one-line demo above shows both
consequences at once.


| format | sign | exponent | mantissa |
|:-------|-----:|---------:|---------:|
| fp32   | 1    | 8        | 23       |
| tf32   | 1    | 8        | 10       |
| bf16   | 1    | 8        | 7        |
| fp16   | 1    | 5        | 10       |
| fp8 (e4m3) | 1 | 4       | 3        |

the figure draws the same table as bit layouts, aligned
at the sign bit so the total widths compare directly: fp32 spends its 32 bits
on a wide mantissa, while bf16 keeps fp32's 8-bit exponent but pays for it
with a narrow 7-bit mantissa, and fp16 inverts the trade.

![Bit layouts of the five formats in the table above, aligned at the sign bit so a wider bar means more total bits: fp32 is the widest, tf32 shares fp32's exponent but truncates the mantissa to 10 bits, and fp16 trades exponent bits for mantissa bits that bf16 spends the other way.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-float-formats.svg)

`finfo` reports what each bit budget buys. Three numbers matter: `max`,
the overflow threshold; `tiny`, the smallest *normal* value, below which
subnormal values provide a short, progressively less precise tail before zero;
and `eps`, the relative step size between adjacent representable values near 1.

In [ ]:
for dtype in (tf.float32, tf.bfloat16, tf.float16):
    fi = ml_dtypes.finfo(dtype.as_numpy_dtype)
    print(f'{dtype.name:10} max {float(fi.max):10.3e}'
          f'  tiny {float(fi.tiny):9.3e}  eps {float(fi.eps):8.3e}')

TensorFlow has no `finfo` of its own (the `tf.experimental.numpy` one chokes
on bfloat16). The numbers live one level down, in `ml_dtypes`, the small
NumPy-extension package that defines the bfloat16 scalar type TensorFlow
depends on; each `tf` dtype hands over its NumPy counterpart via
`as_numpy_dtype`.

bf16 matches fp32's `tiny` and has nearly the same `max`: its shared exponent
width gives the same normal exponent range, while its shorter mantissa makes
the largest finite value slightly smaller. Its `eps` of 0.0078 means two to
three significant decimal digits. fp16 resolves about three to four digits,
overflows at 65504, enters the subnormal range below about
$6.10 \times 10^{-5}$, and reaches zero only below about
$5.96 \times 10^{-8}$. The trade is precision against range, and for deep
learning the choice is lopsided: activations and gradients span many orders
of magnitude, occasional large values are routine, and running out of range
produces `inf` while losing a low-order digit usually costs nothing a noisy
gradient estimate had to offer anyway. That asymmetry is why bf16 became the
default 16-bit format for training.

### TF32: What Happens to fp32 Matrix Multiplication

The second row of the table, tf32, is not a storage dtype; you cannot create a
tensor of it. It is a compute mode of NVIDIA tensor cores (Ampere generation
and later): matrix-multiply inputs are rounded to a 10-bit mantissa while
keeping fp32's exponent, and products are accumulated in fp32. Your tensors
stay fp32; only the arithmetic inside the multiplication runs faster and
slightly less precisely. Most of our frameworks expose a switch for it:

In [ ]:
print(tf.config.experimental.tensor_float_32_execution_enabled())
tf.config.experimental.enable_tensor_float_32_execution(False)
print(tf.config.experimental.tensor_float_32_execution_enabled())
tf.config.experimental.enable_tensor_float_32_execution(True)  # the default

The getter answers `True`: TensorFlow turns tf32 on by default wherever the
hardware supports it, the polarity JAX shares and today's PyTorch inverts,
so an fp32 matrix multiplication on an Ampere-or-later GPU is already taking
the fast path unless you say otherwise. The switch is global, not scoped: a
numerically delicate block disables it, computes, and restores it, as the
cell does. On the CPU this notebook runs on there are no tensor cores and
the flag changes nothing beyond its own value; the distinction takes effect
on the GPUs of that section. For training, the default is
generally safe (products still accumulate in fp32); disable it for
ill-conditioned linear algebra or when reproducing results bit for bit.

### Below 16 Bits

Production inference pushes further down: int8 quantization is standard for
serving, and fp8 training (a 4-bit-exponent variant for the forward pass, a
5-bit-exponent variant for gradients, with per-tensor scaling) runs on
H100-class hardware [@Micikevicius.Stosic.Burgess.ea.2022]. Both require
calibration machinery beyond a dtype argument, so for this book 16 bits is the
floor; the exercises let you inspect the fp8 format's `finfo`.

TensorFlow 2.21 ships no fp8 dtype. The formats exist one level down, in the
same `ml_dtypes` package that supplied our `finfo`
(`ml_dtypes.finfo(ml_dtypes.float8_e4m3fn)` reads them), but no `tf` dtype
wraps them; fp8 work in the TensorFlow world happens in specialized
libraries.

## Dtype Rules: Promotion, Parameters, and Casts

What happens when dtypes meet in one expression? For plain tensors,
TensorFlow refuses to guess. There is no promotion; an operation whose
inputs disagree raises on the spot:

In [ ]:
x16 = tf.ones(3, dtype=tf.float16)
x32 = tf.ones(3, dtype=tf.float32)
try:
    x16 + x32
except tf.errors.InvalidArgumentError as e:
    print(e.message)
(x16 + 1.0).dtype

Only the Python scalar got through: literals like `x + 1.0` are cast to the
tensor's dtype, so sprinkling them into low-precision code is harmless.
Everything else, float with float or float with integer, is an error at the
first operation that touches both. This strictness guards against exactly
the failure that promotion invites in promotion-based libraries, where an fp16
pipeline with a stray fp32 tensor quietly becomes fp32 from that point on,
doubling downstream memory; in TensorFlow the stray tensor cannot travel one
op. The price is that every intended conversion is spelled `tf.cast`.

In [ ]:
net = tf.keras.Sequential([tf.keras.layers.Flatten(),
                           tf.keras.layers.Dense(256, activation='relu'),
                           tf.keras.layers.Dense(10)])
net(tf.zeros((1, 28, 28, 1)))  # build the weights
def param_bytes(net):
    return sum(w.shape.num_elements() * tf.as_dtype(w.dtype).size
               for w in net.weights)
param_bytes(net)

In [ ]:
net = tf.keras.Sequential([
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation='relu', dtype='bfloat16'),
    tf.keras.layers.Dense(10, dtype='bfloat16')])
X = tf.random.normal((8, 28, 28, 1))
print(net(X).dtype, param_bytes(net))

The parameter footprint halves, and note what did not happen: no error,
although we fed an fp32 input to a bf16 model. The strictness of the
previous cell lives at the level of raw operations; a Keras layer casts
floating-point inputs to its `compute_dtype` at the door, so by the time
any op runs, the dtypes already agree. The layer's policy decides the
arithmetic, not the input, and both faces of TensorFlow serve the same end:
ops refuse to guess, layers make the choice explicit at construction.

Casting the model like this is the right tool for *inference*: half the
memory, no gradients to worry about, and a rounding error in the forward pass
rarely changes an argmax. For *training* it is a trap. A single optimizer
update changes a weight by roughly $\eta \cdot g$, often a factor $10^{-4}$
or less of the weight's own magnitude, and adding an increment smaller than
about `eps` times the weight rounds to no change at all. With bf16's `eps` of
0.0078, small updates evaporate and learning stalls; in fp16 the small
gradients themselves flush to zero first. Hence the rule, and it resolves the
single most common confusion in practice:

**Cast the model for inference. For training, keep fp32 weights and run the
compute in 16 bits.**

## Mixed-Precision Training

Mixed-precision training [@Micikevicius.Narang.Alben.ea.2018] splits the
work: parameters stay in fp32 (the *master weights*, so that small updates
still register), while the expensive operations of the forward and backward
pass run in a 16-bit dtype.

In Keras the split is one global switch,
`tf.keras.mixed_precision.set_global_policy('mixed_bfloat16')`. Every layer
constructed afterwards gets the policy pair fp32 `variable_dtype`, bf16
`compute_dtype`: master weights and 16-bit arithmetic, decided per layer
rather than per operation. Two cautions follow from "constructed
afterwards". The policy is consulted when a layer is built, so set it before
creating the model, and it is global state, so we reset it to `'float32'` at
the end of every cell that flips it, or it would silently reshape every
model built later in this notebook. What you compute outside the layers,
such as the loss, follows no policy; we cast logits to fp32 before the loss
for exactly the reason the autocast frameworks pin reductions there.

the figure draws the resulting
loop: this is the distinction that matters between casting a whole model
(everything in one dtype) and mixed precision (fp32 master
weights that a 16-bit forward and backward pass read from and write back to).

![The mixed-precision training loop: fp32 master weights are cast to bf16 for the forward pass and its bf16 activations, the loss accumulates back in fp32, the backward pass produces bf16 gradients, and the optimizer step reads those gradients but updates the fp32 master copy, closing the cycle; the fp16 variant additionally scales the loss up before backward and unscales the gradients back down before the step.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-amp-loop.svg)

In [ ]:
tf.keras.mixed_precision.set_global_policy('mixed_bfloat16')
net = tf.keras.Sequential([tf.keras.layers.Flatten(),
                           tf.keras.layers.Dense(256, activation='relu'),
                           tf.keras.layers.Dense(10)])
Y = net(X)
tf.keras.mixed_precision.set_global_policy('float32')
Y.dtype, net.layers[1].kernel.dtype

Compute in bf16, storage in fp32: exactly the opposite split from
`dtype='bfloat16'`, obtained from one line before construction and nothing
per layer. There is no device argument; the same policy means the same
thing on CPU, GPU, and TPU.

Let us verify the claim that matters, that accuracy survives.
We train the same MLP on Fashion-MNIST twice, once in fp32 and once with
16-bit compute, from the same initialization and on the same batches:

In [ ]:
data = d2l.FashionMNIST(batch_size=256)
images, labels = data.train
batches = [(tf.constant(images[k:k+256, :, :, None], tf.float32) / 255,
            tf.constant(labels[k:k+256], tf.int32))
           for k in range(0, 100 * 256, 256)]
val_images, val_labels = data.val
val_batch = (tf.constant(val_images[:1024, :, :, None], tf.float32) / 255,
             tf.constant(val_labels[:1024], tf.int32))

In [ ]:
def train(mixed):
    tf.keras.utils.set_random_seed(0)  # identical init for both runs
    tf.keras.mixed_precision.set_global_policy(
        'mixed_bfloat16' if mixed else 'float32')
    net = tf.keras.Sequential([tf.keras.layers.Flatten(),
                               tf.keras.layers.Dense(256, activation='relu'),
                               tf.keras.layers.Dense(10)])
    opt = tf.keras.optimizers.SGD(learning_rate=0.1)
    losses = []
    start = time.perf_counter()
    for X, y in batches:
        with tf.GradientTape() as tape:
            logits = tf.cast(net(X), tf.float32)
            loss = tf.reduce_mean(
                tf.keras.losses.sparse_categorical_crossentropy(
                    y, logits, from_logits=True))
        grads = tape.gradient(loss, net.trainable_variables)
        opt.apply_gradients(zip(grads, net.trainable_variables))
        losses.append(float(loss))
    elapsed = time.perf_counter() - start
    Xv, yv = val_batch
    accuracy = float(tf.reduce_mean(tf.cast(
        tf.argmax(net(Xv), axis=1, output_type=yv.dtype) == yv, tf.float32)))
    tf.keras.mixed_precision.set_global_policy('float32')
    return losses, accuracy, elapsed

In [ ]:
loss32, acc32, sec32 = train(mixed=False)
loss16, acc16, sec16 = train(mixed=True)
print(f'fp32: loss {loss32[-1]:.3f}, val acc {acc32:.3f}, {sec32:.2f}s')
print(f'bf16: loss {loss16[-1]:.3f}, val acc {acc16:.3f}, {sec16:.2f}s')
d2l.plot(list(range(1, 101)), [loss32, loss16], 'step', 'loss',
         legend=['fp32', 'mixed_bfloat16'])

The two curves lie on top of each other and their validation accuracies agree:
16-bit rounding perturbs each step slightly, so the trajectories are not
bitwise identical, but they descend at the same rate to the same place. The
reported time reflects this CPU-sized experiment: bf16
may be no faster, and can be slower when casting overhead dominates. The
wall-clock payoff appears on accelerators with native 16-bit matrix units and
in models large enough to occupy them; speedup depends on hardware, shapes,
and the input pipeline (we turn to GPUs in that section). Activations
still occupy half the memory. Note what mixed
precision does *not* buy: the master weights and any Adam state remain fp32,
so the parameter and optimizer terms in the memory arithmetic of
that section do not shrink. The savings are in activations
and speed.


### Loss Scaling for fp16

With bf16, the recipe above is complete. fp16 has one more failure mode, and
it is the opposite end of the axis from the overflow that opened this section:
*gradient underflow*. Many gradients are small. Below about
$6.10 \times 10^{-5}$ fp16 uses increasingly coarse subnormals, and below
about $5.96 \times 10^{-8}$ values round to zero:

In [ ]:
g = tf.constant(1e-8)
tf.cast(g, tf.float16), tf.cast(g * 2**16, tf.float16)

The gradient vanishes, yet the same value scaled by $2^{16}$ is perfectly
representable. That is the idea of *loss scaling*: multiply the loss by a
large constant before backpropagation, so that by linearity every gradient is
scaled into representable territory, then divide the gradients by the same
constant before the optimizer step.

In Keras loss scaling is the `'mixed_float16'` policy plus an optimizer
wrapper. Under that policy, `model.compile` wraps whatever optimizer you
pass in `tf.keras.mixed_precision.LossScaleOptimizer`, which chooses the
scale dynamically: start high, shrink when scaled gradients overflow to
`inf` (skipping that step), grow back periodically. `model.fit` users
therefore get correct fp16 training without touching anything; a custom
loop applies the wrapper itself, multiplies with its `scale_loss` before
taking gradients, and lets the wrapped optimizer unscale and skip bad steps.
None of this machinery exists for `'mixed_bfloat16'` because none is needed:
bf16's normal exponent range matches fp32 and avoids fp16's narrow-range
failure in ordinary training. Hence the modern default our training cell followed: prefer
`'mixed_bfloat16'` where the hardware supports it, and reach for
`'mixed_float16'` plus the scaler only on older accelerators.

## When Numerics Bite

A short field guide for the day training misbehaves.

**The loss is NaN.** NaN is usually a symptom, not the disease; the disease is
`inf`, because `inf` arithmetic breeds NaN:

In [ ]:
s = tf.constant(60000., dtype=tf.float16) * 2  # overflows
s, s - s

By the time a NaN reaches your loss, the overflow that spawned it may be many
operations upstream, and once a NaN lands in the weights it poisons every
subsequent step. So diagnose in order: first check ranges (is anything in
fp16? are intermediate values in the $10^4$ regime and headed for the 65504
ceiling?), and only then blame the learning rate.

TensorFlow can localize the culprit for you: call
`tf.debugging.enable_check_numerics()` and execution stops with an
`InvalidArgumentError` at the first operation whose output contains `inf` or
NaN, instead of letting it wash downstream into the loss. The check wraps
every operation, so treat it as a debugging mode, not a default.

**Let the library take the log.** The naive evaluation of
$\log \sum_i \exp(x_i)$ overflows long before the answer does:

In [ ]:
x = tf.constant([12.0, 11.0, 10.0], dtype=tf.float16)
tf.math.log(tf.reduce_sum(tf.exp(x))), tf.math.reduce_logsumexp(x)

The answer, 12.4, is unremarkable; only the intermediate $e^{12}$ exceeds
65504. The subtract-the-max shift from that section fixes
it, and the practical form of the lesson is to never hand-roll the pattern.

`tf.math.reduce_logsumexp`, `tf.nn.log_softmax`, and Keras's cross-entropy
losses with `from_logits=True` all build the shift in.

**Accumulate in fp32.** Long sums in a 16-bit dtype drift, because once the
running total is large, each small increment falls below the spacing of
representable values and is partly or wholly rounded away:

In [ ]:
x = tf.fill([10**7], tf.constant(1e-3, tf.float16))
(tf.cumsum(x)[-1], tf.cumsum(tf.cast(x, tf.float32))[-1],
 tf.cumsum(tf.cast(x, tf.float64))[-1])

TensorFlow's implementation loses enough fp16 increments to land at 8192,
well below the 10004.04 sum of the stored values. Its fp32 accumulation reaches
10004.04 to the displayed precision, and fp64 supplies the reference value.
The result is implementation-dependent because a parallel prefix sum groups
terms differently from a left-to-right loop, but the lesson is stable: the
fp16 accumulator loses small increments as partial sums grow. Means over large
batches, epoch-level loss totals, and variance computations all follow this
pattern. Choose an accumulator wide relative to the length of the sum, and
prefer `tf.reduce_sum` when you need only the total so the library can use an
accurate reduction tree rather than materializing every prefix.

Finally, do not expect bitwise equality across
numeric configurations: tf32 versus fp32, or mixed precision on versus off,
differ in the last bits by design. What reproducibility you can demand, and
how to get it, is the subject of that section.

## Summary

A dtype is a contract about range and precision, and the 16-bit formats split
the difference between them: fp16 keeps precision and forfeits range, bf16
keeps fp32's range and forfeits precision, which suits deep learning better.

TensorFlow's raw ops never promote; a dtype mismatch raises at the first
operation, and Keras layers resolve this by casting inputs to their dtype
policy, the pair of storage and compute formats fixed at construction.
`dtype='bfloat16'` sets both and casts a model for inference;
`set_global_policy('mixed_bfloat16')` keeps fp32 master weights over bf16
compute, mixed-precision training as one line of configuration (reset the
policy when you are done, it is global). fp16 training additionally needs
the loss scaling that `LossScaleOptimizer` automates under
`'mixed_float16'`; bf16 does not.

When numbers misbehave: check for
overflow before blaming the learning rate, use the library's `logsumexp`
family, and accumulate long sums in fp32.

## Exercises

1. Redo the memory arithmetic of that section for
   mixed-precision training with Adam: fp32 master weights, fp32 gradients,
   two fp32 moment estimates, and bf16 activations. Which term dominates now,
   and how does the total compare with all-fp32 training?
1. Time the fp32 and 16-bit runs of `train` against each other while
   shrinking the hidden width and the batch size. Find a model small enough
   that mixed precision is *slower* than fp32, and explain where the
   crossover comes from.
1. Print every field of `torch.finfo(torch.float8_e4m3fn)` (in JAX,
   `jnp.finfo(jnp.float8_e4m3fn)`; in TensorFlow,
   `ml_dtypes.finfo(ml_dtypes.float8_e4m3fn)`; MXNet has no fp8 dtype, so
   borrow the standalone `ml_dtypes` package) and compare with fp16 and
   bf16. Explain
   the name `e4m3fn`, including why its `max` is 448 rather than the value
   the exponent bits alone would suggest.
1. Under autocast, normalization layers run in fp32. To see why, take the
   `RMSNorm` layer of that section, feed it inputs with a
   standard deviation of about 100, and force the computation to fp16. Which
   intermediate quantity fails first?